In [127]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage, ToolCall, ToolMessage
from typing import Literal 
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

In [40]:
load_dotenv()

True

In [67]:
chatModel = ChatOpenAI(model='gpt-4o')

In [150]:
@tool
def temperature_tool(place: str)-> float:
    """Tool that fetches temperature of any location in the world"""
    return 20.0


@tool
def calculatorTool(a: int, b: int, runTimeVariable: ToolRuntime[int], op: Literal['add', 'sub', 'mul', 'divide'])-> float:
    """Tool that fetches temperature of any location in the world"""
    return a + b + runTimeVariable.context # you should access context property using .context of toolRuntime

In [151]:
temperature_tool.invoke("apple")

20.0

In [152]:
toolCall: ToolMessage = ToolCall(name='temperature_tool', args={"place": "apple"}, id="call_1",          # any string; becomes ToolMessage.tool_call_id
    type="tool_call",     # this is what makes .invoke() treat it as a tool call
)

In [144]:
temperature_tool.invoke(toolCall)

ToolMessage(content='20.0', name='temperature_tool', tool_call_id='call_1')

In [ ]:
checkpointer = InMemorySaver()
agent = create_agent(model = chatModel,
                      tools=[temperature_tool, calculatorTool],
                     checkpointer=checkpointer,
                     system_prompt=" Never guess a value that another tool can return; wait for that tool's result before using it.", 
)

In [ ]:
config = {'configurable': {'thread_id': '1'}}
response = agent.invoke({"messages": "What is the temperature of india and multily that value with 3"}, config= config, context=100)

In [147]:
response['messages'][1].tool_calls

[{'name': 'temperature_tool',
  'args': {'place': 'india'},
  'id': 'call_c3tgAESuG8sSq4OY5RUuzk8g',
  'type': 'tool_call'}]

In [148]:
response

{'messages': [HumanMessage(content='What is the temperature of india and multily that value with 3', additional_kwargs={}, response_metadata={}, id='9f6d4cc4-2ea1-4bb1-9e50-a517fd1eac76'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 131, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6a23e70e3e', 'id': 'chatcmpl-EZUqVuUyyAGFJtEXO1zJIDCnOiqr5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a14332-b5e2-7660-95ba-5469629b44c3-0', tool_calls=[{'name': 'temperature_tool', 'args': {'place': 'india'}, 'id': 'call_c3tgAESuG8sSq4OY5RUuzk8g', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m

In [ ]:
# streaming

streamResponse =  agent.stream({"messages": "What is the temperature of india and multily that value with 3"}, config= config, stream_mode='updates')

In [123]:
for res in streamResponse: # res is of type dict.
    for model, message in res.items():
        print(message['messages'])

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 131, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6a23e70e3e', 'id': 'chatcmpl-EZUjQLMYdNhRivvyh8yaTU9JmwbVg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1432c-015e-73a0-9fc8-9431430fb781-0', tool_calls=[{'name': 'temperature_tool', 'args': {'place': 'India'}, 'id': 'call_TNs2TD7O05rWTRFwnPOFvqAI', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 14, 'total_tokens': 145, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]
[ToolMes